In [2]:
# setup
import io
import base64
import sys
import glob
import errno
from collections import defaultdict
import os
import gc

import numpy as np
import cupy as cp
import h5py
import scipy as sp
import pandas as pd
import itertools
import multiprocessing as mproc
import pandas as pd
import PIL as pil
import rembg
import glob

sys.path.append(os.getcwd())

%reload_ext autoreload
%autoreload 2

from IPython.display import display, HTML, Math, Latex
display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

import matplotlib.pyplot as plt
# from mpl_toolkits.axes_grid1.axes_divider import make_axes_locatable
import matplotlib as mpl
# from mpl_toolkits.mplot3d import Axes3D

%matplotlib inline
#%matplotlib notebook
    
# mpl.rcParams['text.usetex'] = 'True'
mpl.rcParams['axes.grid'] = False
mpl.rcParams['xtick.top'] = True
mpl.rcParams['xtick.bottom'] = True
mpl.rcParams['ytick.left'] = True
mpl.rcParams['ytick.right'] = True
mpl.rcParams['xtick.minor.visible'] = True
mpl.rcParams['ytick.minor.visible'] = True
mpl.rcParams['xtick.direction'] = 'in'
mpl.rcParams['ytick.direction'] = 'in'
mpl.rcParams['xtick.major.size'] = 14
mpl.rcParams['ytick.major.size'] = 14
mpl.rcParams['xtick.minor.size'] = 7
mpl.rcParams['ytick.minor.size'] = 7
mpl.rcParams['xtick.major.width'] = 2
mpl.rcParams['ytick.major.width'] = 2
mpl.rcParams['xtick.minor.width'] = 1.2
mpl.rcParams['ytick.minor.width'] = 1.2
mpl.rcParams['axes.labelsize'] = 20
mpl.rcParams['xtick.labelsize'] = 20
mpl.rcParams['ytick.labelsize'] = 20
mpl.rcParams['legend.loc'] = 'best'
mpl.rcParams['legend.fontsize'] = 25
mpl.rcParams['lines.linewidth'] = 2
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['font.serif'] = 'Computer Modern'
mpl.rcParams['xtick.major.pad']='8'
mpl.rcParams['ytick.major.pad']='8'
mpl.rcParams['ytick.major.pad']='8'

blue = '#1f77b4'
orange = '#ff7f0e'
green = '#2ca02c'
red = '#ad494a'
violet = '#9467bd'
brown = '#8c564b'

In [ ]:
class database():

    def __init__(self, name):
        
        self.name = name
        self.dataframe = pd.read_excel(self.name)
        self.dataframe.fillna('', inplace=True)

    def is_hybrid(self):
        return self.dataframe[self.dataframe['Hybrid or OP'].str.contains("hybrid")].sort_index()

    def is_allyear(self):
        return self.dataframe[self.dataframe['Sowing and transplant time'].str.contains("all year")].sort_index()
    
    def is_crop(self, name):
        if not isinstance(name, str):
            raise TypeError("name should be a string")
        
        df1 = self.dataframe[self.dataframe['Crop'].str.contains(name)]
        df2 = self.dataframe[self.dataframe['Crop'].str.contains(name.lower())]
        df3 = self.dataframe[self.dataframe['Crop'].str.contains(name.capitalize())]
        return pd.concat([df1, df2, df3], axis=0).drop_duplicates().sort_index()
    
    def is_heat_tolerant(self):
        return self.dataframe[self.dataframe['Weather Tolerance'].str.contains('heat')]
    
    def is_rain_tolerant(self):
        return self.dataframe[self.dataframe['Weather Tolerance'].str.contains('rain')]
    
    def is_transport_storage_good(self):
        return self.dataframe[self.dataframe['Transport and storage property'].str.contains('good')]
    
    def is_maturity_within(self, days):
        lower_bounds = self.dataframe['Maturity (days)'].apply(
            lambda x: x if isinstance(x, int) else int(x.split('-')[0])
        )
        return self.dataframe[(lower_bounds <= days).tolist()]

In [198]:
db = database('data/Tomato.xlsx')

In [199]:
db.is_maturity_within(60)


,Variety,Crop,Hybrid or OP,Sowing and transplant time,Maturity (days),After sowing or transplant,Weight (g),Fruit Size,"Fruit shape, type, color",Disease Tolerance,Weather Tolerance,Transport and storage property
0,F1 Hybrid Tomato Beautiful,Tomato,hybrid,all year,50-55,transplant,150,,oval,"BW, Nematode, TYLCV",heat tolerant,good
4,F1 Hybrid Tomato Ruposhi Plus,Tomato,hybrid,August to October,55-60,transplant,100-110,,firm,"BW, TYLCV",heat tolerant,good
5,F1 Hybrid Tomato Golden Cherry,Tomato,hybrid,August to October,50-55,transplant,30,,"compact, yellow","BW, TYLCV",,
6,F1 Hybrid Tomato Profit Early,Tomato,hybrid,May to August,50,transplant,135,,"oval, firm","BW, FW, TYLCV, TMV",heat tolerant,good
7,F1 Hybrid Tomato Red Cherry,Tomato,hybrid,August to October,45,transplant,20-22,,"firm, bright-red",,,good
8,F1 Hybrid Tomato Red King,Tomato,hybrid,Summer,60,transplant,100,6.1 x 5.4 cm,"flattened-rounded, bright-red","BW, TYLCV",,
9,F1 Hybrid Tomato Lovely,Tomato,hybrid,August to October,50,transplant,130,,"firm, round","BW, Nematode, TYLCV",,good
10,F1 Hybrid Tomato Red Beauty-1,Tomato,hybrid,Summer,50-65,transplant,135,6.4 x 5.3 cm,"flattened-rounded, red","BW, TYLCV",heat and rain tolerant,
11,F1 Hybrid Tomato Super,Tomato,hybrid,July to October,55-66,transplant,150-200,,"oval , bright red, firm","BW, TMV",heat and rain tolerant,good


In [169]:
db.dataframe

,Variety,Crop,Hybrid or OP,Sowing and transplant time,Maturity (days),After sowing or transplant,Weight (g),Fruit Size,"Fruit shape, type, color",Disease Tolerance,Weather Tolerance,Transport and storage property
0,F1 Hybrid Tomato Beautiful,Tomato,hybrid,all year,50-55,transplant,150,,oval,"BW, Nematode, TYLCV",heat tolerant,good
1,F1 Hybrid Tomato Beautiful-2,Tomato,hybrid,all year,65,transplant,135,5.7 x 7.2 cm,oval,"BW, FW, TYLCV, LB",,good
2,F1 Hybrid Tomato Sumo,Tomato,hybrid,November onward,70,transplant,200-250,,compact,Disease tolerant variety,,good
3,F1 Hybrid Tomato Wonderful Plus,Tomato,hybrid,August to November,70-75,transplant,140-150,,firm,"BW, FW, LB, TYLCV",,good
4,F1 Hybrid Tomato Ruposhi Plus,Tomato,hybrid,August to October,55-60,transplant,100-110,,firm,"BW, TYLCV",heat tolerant,good
5,F1 Hybrid Tomato Golden Cherry,Tomato,hybrid,August to October,50-55,transplant,30,,"compact, yellow","BW, TYLCV",,
6,F1 Hybrid Tomato Profit Early,Tomato,hybrid,May to August,50,transplant,135,,"oval, firm","BW, FW, TYLCV, TMV",heat tolerant,good
7,F1 Hybrid Tomato Red Cherry,Tomato,hybrid,August to October,45,transplant,20-22,,"firm, bright-red",,,good
8,F1 Hybrid Tomato Red King,Tomato,hybrid,Summer,60,transplant,100,6.1 x 5.4 cm,"flattened-rounded, bright-red","BW, TYLCV",,
9,F1 Hybrid Tomato Lovely,Tomato,hybrid,August to October,50,transplant,130,,"firm, round","BW, Nematode, TYLCV",,good
